# Inference Demo (CKPT3 Stacking)


This notebook loads saved CKPT3 artifacts and runs inference on a small batch.

If saved splits or checkpoints are missing, it rebuilds them automatically.


In [10]:
import sys
from pathlib import Path

# Make repo root importable
sys.path.append('..')


In [11]:
import pandas as pd
from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi

DATA_DIR = Path('../data/ckpt2_splits')
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / 'train_features.csv'
val_path = DATA_DIR / 'val_features.csv'
test_path = DATA_DIR / 'test_features.csv'

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    print('Saved splits missing. Rebuilding splits...')
    file_2009 = Path('../data/Year 2009-2010.csv')
    file_2010 = Path('../data/Year 2010-2011.csv')
    if not (file_2009.exists() and file_2010.exists()):
        raise FileNotFoundError('Raw dataset files not found in ../data/')

    df_raw = load_online_retail_ii(file_2009, file_2010)
    df_clean = clean_data(df_raw)

    train_cutoffs = ['2010-06-01','2010-09-01','2010-12-01','2011-03-01']
    val_cutoff = '2011-06-01'
    test_cutoff = '2011-09-01'

    train_df, val_df, test_df = create_temporal_splits_multi(
        df_clean, train_cutoffs, val_cutoff, test_cutoff, obs_months=6, horizon_months=3
    )

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)
    print('Splits saved to', DATA_DIR)

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Default feature list (keeps CustomerID if present)
exclude = {'customer_id','target','cutoff_date','horizon_start','horizon_end'}
default_feature_cols = [c for c in test_df.columns if c not in exclude]


In [12]:
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from src.stacking import StackedEnsemble

CKPT_DIR = Path('../checkpoints/ckpt3/config_b')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not (CKPT_DIR / 'base_models.pkl').exists():
    print('Checkpoint missing. Training CKPT3 Config B...')

    feature_cols = default_feature_cols
    X_train = train_df[feature_cols]
    y_train = train_df['target']

    base_models = {
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    }
    try:
        import xgboost as xgb
        base_models['XGBoost'] = xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
    except Exception:
        print('XGBoost not available; using EN + RF only.')

    stack_demo = StackedEnsemble(use_features=True, n_folds=5)
    oof_demo = stack_demo.generate_oof_predictions(X_train, y_train, base_models)
    stack_demo.train(X_train, y_train, oof_demo, save_dir=str(CKPT_DIR), save_models=True)
    stack_demo.train_base_models_final(X_train, y_train, base_models)

    print('Checkpoint created.')
else:
    print('Checkpoint found.')

# Load checkpoint
stack_loaded = StackedEnsemble(use_features=True, n_folds=5)
stack_loaded.load_models(str(CKPT_DIR))

# Determine feature columns from saved model if available
if hasattr(stack_loaded, 'base_models') and len(stack_loaded.base_models) > 0:
    sample_model = list(stack_loaded.base_models.values())[0]
    if hasattr(sample_model, 'feature_names_in_'):
        feature_cols = list(sample_model.feature_names_in_)
    else:
        feature_cols = default_feature_cols
else:
    feature_cols = default_feature_cols

# Small batch inference
batch = test_df.sample(20, random_state=42).copy()
X_batch = batch[feature_cols]
y_batch = batch['target'].values

pred_batch, _ = stack_loaded.predict(X_batch)

demo_df = pd.DataFrame({
    'customer_id': batch.get('CustomerID', batch.index),
    'true_target': y_batch,
    'pred_stack': np.round(pred_batch, 3)
})

print(demo_df.to_string(index=False))

OUT = Path('../results/demo_ckpt3_inference.csv')
OUT.parent.mkdir(exist_ok=True)
demo_df.to_csv(OUT, index=False)
print('Saved inference log to', OUT)


Checkpoint found.
✓ Loaded models from ../checkpoints/ckpt3/config_b
 customer_id  true_target  pred_stack
     17954.0            1       2.155
     12447.0            0       0.396
     16552.0            0       0.325
     14690.0            1       0.416
     15059.0            2       1.619
     16016.0            3       0.864
     13835.0            2       0.712
     17653.0            3       0.362
     17097.0            1       1.848
     15298.0            5       1.131
     13950.0            3       0.355
     17188.0            2       0.790
     16303.0            1       0.325
     14231.0            0       0.753
     15671.0            3       1.328
     14619.0            0       0.308
     17582.0            0       0.180
     16549.0            8       0.313
     16477.0            1       0.938
     14866.0            8       1.325
Saved inference log to ../results/demo_ckpt3_inference.csv


We are predicting customer purchase count in the future horizon.

Specifically:

Target = number of unique invoices (transactions) for each customer

Horizon window = next 3 months after the cutoff date

So the model predicts how many transactions a customer will make in the next 3 months.

That’s why predictions are non‑integer expected counts (e.g., 1.32 means “about 1–2 purchases”).